# Replication: "Smart Green Nudging" — von Zahn et al. (2024)
> *Marketing Science* — https://pubsonline.informs.org/doi/10.1287/mksc.2022.0393

**Sections**
1. Setup & Data Loading
2. Digital Footprint Feature Engineering
3. ATE Estimation (OLS · IPW · AIPW)
4. Robustness Checks (permutation test · bandwidth sensitivity)
5. Causal Forest — Heterogeneous Treatment Effects (CATE)
6. Nudge Targeting Policy & Profit Analysis
7. Summary & Conclusions

All heavy logic lives in `src/` — each cell below is a **one-liner import + call**.

---
> **Note:** This replication uses synthetic data generated to match the paper's
> described data-generating process. The original retailer data is proprietary.

## 0. Install Dependencies
```bash
pip install econml litellm pandas numpy matplotlib seaborn scikit-learn statsmodels
# Ollama (for LLM cells): https://ollama.ai  →  ollama pull llama3.2
```

---
## 1. Setup & Data Loading

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, 'src')          # make helper modules importable

import numpy as np
import pandas as pd
from pathlib import Path

from features       import engineer_features, get_feature_cols, check_covariate_balance
from ate            import run_ate_suite, permutation_test, bandwidth_sensitivity
from causal_forest  import fit_causal_forest, compute_rate
from plots          import *
from llm            import print_interpretation, conclude

RANDOM_SEED   = 42
TREATMENT_COL = 'treatment'
OUTCOME_COL   = 'returned'
np.random.seed(RANDOM_SEED)

In [ ]:
DATA_DIR   = Path('data')
df_train   = pd.read_csv(DATA_DIR / 'train.csv')
df_test    = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train: {df_train.shape}  |  Test: {df_test.shape}')
df_train.head()

In [ ]:
plot_descriptive_overview(df_train, TREATMENT_COL, OUTCOME_COL);

In [ ]:
rc = df_train[df_train[TREATMENT_COL]==0][OUTCOME_COL].mean()
rt = df_train[df_train[TREATMENT_COL]==1][OUTCOME_COL].mean()
print_interpretation(
    'Data Loading & Descriptive Overview',
    f'N train={len(df_train)}, N test={len(df_test)}.\n'
    f'Control return rate={rc:.3f}, Nudge return rate={rt:.3f}.\n'
    f'Raw difference={rt-rc:+.4f} ({(rt-rc)*100:+.2f} pp).'
)

---
## 2. Digital Footprint Feature Engineering

The paper constructs behavioural features from customers' digital footprints —
browsing sessions, past return history, basket composition, and device usage.
Feature logic is in `src/features.py`.

In [ ]:
df_train = engineer_features(df_train)
df_test  = engineer_features(df_test)

FEATURE_COLS = get_feature_cols(df_train, exclude=[TREATMENT_COL, OUTCOME_COL])
print(f'{len(FEATURE_COLS)} numeric features after engineering:')
print(FEATURE_COLS)
df_train[FEATURE_COLS].describe().round(3)

In [ ]:
plot_correlation_heatmap(df_train, FEATURE_COLS, OUTCOME_COL);

In [ ]:
# Covariate balance check — confirms randomisation quality
balance = check_covariate_balance(df_train, TREATMENT_COL, FEATURE_COLS)
print(balance.to_string(index=False))
plot_covariate_balance(balance);

In [ ]:
top_corr = df_train[FEATURE_COLS + [OUTCOME_COL]].corr()[OUTCOME_COL].drop(OUTCOME_COL)
top_corr = top_corr.sort_values(key=abs, ascending=False)
print_interpretation(
    'Feature Engineering',
    f'{len(FEATURE_COLS)} features engineered.\n'
    f'Top correlates with return outcome:\n{top_corr.head(5).to_string()}'
)

---
## 3. ATE Estimation — Field Experiment

We estimate the **Average Treatment Effect (ATE)** using four estimators:
OLS (naive), OLS + covariates (HC3), IPW (Horvitz-Thompson), and AIPW
(doubly-robust). All logic is in `src/ate.py`.

In [ ]:
X_train = df_train[FEATURE_COLS].fillna(0)
T_train = df_train[TREATMENT_COL]
Y_train = df_train[OUTCOME_COL]

X_test  = df_test[FEATURE_COLS].fillna(0)
T_test  = df_test[TREATMENT_COL]
Y_test  = df_test[OUTCOME_COL]

ate_results = run_ate_suite(Y_train, T_train, X_train, n_boot=500, seed=RANDOM_SEED)
print(ate_results.summary())

In [ ]:
plot_ate_forest(ate_results.to_dataframe());

In [ ]:
df_ate = ate_results.to_dataframe()
print_interpretation(
    'ATE Estimation (OLS · IPW · AIPW)',
    df_ate[['method','ate','pvalue']].to_string(index=False)
)

---
## 4. Robustness Checks

### 4.1 Permutation / Placebo Test
Shuffles the treatment label 1,000 times to construct the null distribution.
The empirical p-value is the fraction of shuffles with |ATE| ≥ |observed ATE|.

In [ ]:
perm = permutation_test(Y_train, T_train, X_train, n_permutations=1_000, seed=RANDOM_SEED)
print(f"Observed ATE : {perm['observed_ate']:+.4f}")
print(f"Null mean    : {perm['null_mean']:+.4f}  std={perm['null_std']:.4f}")
print(f"Empirical p  : {perm['p_value']:.4f}")
plot_permutation_null(perm);

### 4.2 Bandwidth / Sample-Restriction Sensitivity
Re-estimates OLS+controls ATE at progressively restricted subsamples
(100%, 90%, 75%, … 15%). A stable ATE across bandwidths supports robustness.

In [ ]:
bw_df = bandwidth_sensitivity(
    df_train, Y_col=OUTCOME_COL, T_col=TREATMENT_COL, feature_cols=FEATURE_COLS
)
print(bw_df[['bandwidth','n','ate','ci_lo','ci_hi','pvalue','sig']].to_string(index=False))
plot_bandwidth_sensitivity(bw_df);

In [ ]:
print_interpretation(
    'Robustness Checks',
    f"Permutation p-value: {perm['p_value']:.4f}.\n"
    f"ATE stable across bandwidths: "
    f"{'YES' if bw_df['ate'].std() < 0.005 else 'REVIEW'}.\n"
    f"Bandwidth results:\n{bw_df[['bandwidth','ate','pvalue']].to_string(index=False)}"
)

---
## 5. Causal Forest — Heterogeneous Treatment Effects (CATE)

We use `CausalForestDML` (Wager & Athey 2018) to estimate individual-level
treatment effects. The BLP calibration test checks whether heterogeneity is
statistically significant. All logic is in `src/causal_forest.py`.

In [ ]:
cf = fit_causal_forest(
    Y_train, T_train, X_train, X_test,
    feature_names=FEATURE_COLS,
    n_estimators=2_000,
    seed=RANDOM_SEED,
)
print(cf.summary())

In [ ]:
df_test = df_test.copy()
df_test['cate']    = cf.cate
df_test['cate_lb'] = cf.cate_lb
df_test['cate_ub'] = cf.cate_ub

plot_cate_distribution(cf);

In [ ]:
plot_feature_importance(cf.feat_imp);

In [ ]:
# RATE (Rank Average Treatment Effect) — targeting value test
rate_res = compute_rate(cf.cate, Y_test.values, T_test.values)
print(f"RATE = {rate_res['rate']:+.4f}  (positive → smart targeting beats random)")
plot_toc_curve(rate_res);

In [ ]:
pct_benefit = (cf.cate < 0).mean()
print_interpretation(
    'Causal Forest / CATE',
    f"Mean CATE={cf.cate.mean():+.4f}, std={cf.cate.std():.4f}.\n"
    f"{pct_benefit:.1%} of test customers have CATE<0 (nudge reduces returns).\n"
    f"Confidence score (CI excludes 0): {cf.conf_score:.1%}.\n"
    f"RATE={rate_res['rate']:+.4f}.\n"
    f"BLP test: {cf.blp_test}\n"
    f"Top heterogeneity drivers: {cf.feat_imp.head(5).index.tolist()}"
)

---
## 6. Nudge Targeting Policy & Profit Analysis

We sweep the targeting threshold from 0%→100% and compare profit gains
under **smart targeting** (CATE-ranked) vs. **universal nudging** (ATE-based).

In [ ]:
RETURN_COST     = 15.0    # € cost to retailer per returned item
NUDGE_COST      = 0.10    # € cost per nudge shown
ATE_CTRL        = ate_results.ate[1]   # OLS + controls ATE

df_pol = df_test.sort_values('cate').reset_index(drop=True)
n      = len(df_pol)
fracs  = np.linspace(0, 1, 101)

profit_smart, profit_univ = [], []
for frac in fracs:
    k        = int(frac * n)
    averted  = (-df_pol['cate'].iloc[:k].clip(upper=0)).sum()
    profit_smart.append(averted * RETURN_COST - k * NUDGE_COST)
    averted_u = frac * n * max(0, -ATE_CTRL)
    profit_univ.append(averted_u * RETURN_COST - k * NUDGE_COST)

profit_smart  = np.array(profit_smart)
profit_univ   = np.array(profit_univ)
best_frac     = fracs[np.argmax(profit_smart)]

print(f'Optimal targeting share : {best_frac:.1%}')
print(f'Max profit — smart      : €{profit_smart.max():.2f}')
print(f'Max profit — universal  : €{profit_univ.max():.2f}')
print(f'Value of personalisation: €{profit_smart.max() - profit_univ.max():.2f}')

plot_policy_curve(fracs, profit_smart, profit_univ, best_frac);

In [ ]:
df_test['cate_quartile'] = pd.qcut(
    df_test['cate'], q=4,
    labels=['Q1\n(most\nreceptive)', 'Q2', 'Q3', 'Q4\n(least\nreceptive)']
)
seg = df_test.groupby('cate_quartile', observed=True).agg(
    n=          (OUTCOME_COL, 'count'),
    return_rate=(OUTCOME_COL, 'mean'),
    mean_cate=  ('cate',      'mean'),
).round(4)
print(seg)
plot_cate_segments(seg);

In [ ]:
print_interpretation(
    'Nudge Targeting Policy',
    f"Optimal share={best_frac:.1%}, smart gain=€{profit_smart.max():.2f}, "
    f"universal gain=€{profit_univ.max():.2f}, "
    f"personalisation value=€{profit_smart.max()-profit_univ.max():.2f}.\n"
    f"Segment breakdown:\n{seg.to_string()}"
)

---
## 7. Summary & Conclusions

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'ATE — OLS naive',
        'ATE — OLS + controls',
        'ATE — IPW',
        'ATE — AIPW (doubly-robust)',
        'Permutation p-value',
        'Mean CATE (causal forest)',
        'CATE std (heterogeneity)',
        '% customers CATE < 0',
        'Confidence score',
        'RATE',
        'Optimal targeting share',
        'Max profit — smart targeting',
        'Max profit — universal nudging',
        'Value of personalisation',
    ],
    'Value': [
        f"{ate_results.ate[0]:+.4f}  (p={ate_results.pvalue[0]:.4f})",
        f"{ate_results.ate[1]:+.4f}  (p={ate_results.pvalue[1]:.4f})",
        f"{ate_results.ate[2]:+.4f}  (p={ate_results.pvalue[2]:.4f})",
        f"{ate_results.ate[3]:+.4f}  (p={ate_results.pvalue[3]:.4f})",
        f"{perm['p_value']:.4f}",
        f"{cf.cate.mean():+.4f}  ({cf.cate.mean()*100:+.2f} pp)",
        f"{cf.cate.std():.4f}",
        f"{pct_benefit:.1%}",
        f"{cf.conf_score:.1%}",
        f"{rate_res['rate']:+.4f}",
        f"{best_frac:.1%}",
        f"€{profit_smart.max():.2f}",
        f"€{profit_univ.max():.2f}",
        f"€{profit_smart.max() - profit_univ.max():.2f}",
    ]
})

print('═' * 65)
print('  REPLICATION SUMMARY')
print('═' * 65)
print(summary.to_string(index=False))
print('═' * 65)

In [ ]:
print(conclude(summary))

---
## References

- von Zahn, M., Feuerriegel, S., & Kuehl, N. (2024). **Smart Green Nudging: Reducing Product Returns Through Digital Footprints and Causal Machine Learning.** *Marketing Science*. https://doi.org/10.1287/mksc.2022.0393
- Wager, S., & Athey, S. (2018). Estimation and inference of heterogeneous treatment effects using random forests. *JASA*, 113(523), 1228–1242.
- Chernozhukov, V. et al. (2022). Generic Machine Learning Inference on Heterogeneous Treatment Effects in Randomised Experiments. *arXiv:1712.04802*.
- EconML: https://econml.azurewebsites.net/
- LiteLLM: https://docs.litellm.ai/